In [ ]:
from pylab import *
import os
from scipy.stats import linregress as linregress

# Resolution Convergence

## Simulation Setup

We have the following $N = N_\rho = N_z$ number of interior points in each dimension.

In [ ]:
N = np.logspace(7, 10, 12, base = 2.0, dtype = int)
print(N)

For variable resolution $\Delta \rho = \Delta z$ we have to fix $\omega$ and the grid size. We will work with a constant $r_\infty$.

In [ ]:
rr_inf = 1.0 * (N[0] + 3 - 2 + 0.5)
rr_inf

Now that we have our grid extension, we calculate the resolution for each $N$. Since $\rho = \Delta \rho\,(i - g + 1/2)$ for $i = 0,\,\ldots,\,N_\rho + 2g - 1$, with $g$ the number of ghost points (2 for 4th order discretization), we have

In [ ]:
ghost = 2
dr = rr_inf / (N + 2 * ghost - 1 - ghost + 0.5)
print(dr)

Print our working resolution and number of interior points in a readable format.

In [ ]:
["%.5E and %04d" % (dr[i], N[i]) for i in range(len(N))]

Next, generate the parameter files. Write them to a local directory and then `rsync` everything to the cluster to run. Notice that the first file for $N = 128$, $\Delta \rho = 1$ is not included. Since this is the "seed" simulation, this must be generated manually or previously.

In [ ]:
# Directory names.
local_directory = "/mnt/e/RBS/Regularized Convergence/RConvergence/"
simul_directory = "/storage/icn/sontanon/out/RConvergence/"

# Fixed omega.
w = 0.75000

# Other parameters.
order = 4
l = 2

# Solver parameters.
solverType    = 1
localSolver   = 1
epsilon       = 2.0E-13
maxNewtonIter = 50
lambda0       = 1.0E-01
lambdaMin     = 1.0E-05
useLowRank    = 0

In [ ]:
# Generate parameter files. Notice that N[0] is excluded.
for i in range(1, len(N)):
    # Write parameter name.
    par_name = "%sl=%d,%.5E,dr=%.5E,N=%04d.par" % (local_directory, l, w, dr[i], N[i])
    
    # Open file.
    f = open(par_name, "w")
    
    # Generate parameter file string.
    par_file = """# GRID
dr 		= %.16E
dz 		= %.16E
NrInterior	= %d
NzInterior	= %d
order 		= %d

# SCALAR FIELD PROPERTIES
l	 = %d
m 	 = 1.0

# INITIAL DATA
readInitialData	= 3
NrTotalInitial 	= %d
NzTotalInitial 	= %d
dr_i		= %.16E
dz_i		= %.16E
ghost_i		= %d
order_i		= %d
log_alpha_i 	= "%sl=%d,w=%.5E,dr=%.5E,N=%04d/log_alpha_f.asc"
beta_i		= "%sl=%d,w=%.5E,dr=%.5E,N=%04d/beta_f.asc"
log_h_i		= "%sl=%d,w=%.5E,dr=%.5E,N=%04d/log_h_f.asc"
log_a_i		= "%sl=%d,w=%.5E,dr=%.5E,N=%04d/log_a_f.asc"
psi_i		= "%sl=%d,w=%.5E,dr=%.5E,N=%04d/psi_f.asc"
lambda_i	= "%sl=%d,w=%.5E,dr=%.5E,N=%04d/lambda_f.asc"
w_i		= "%sl=%d,w=%.5E,dr=%.5E,N=%04d/w_f.asc"

# FIXED VARIABLE.
fixedPhi  	= 0
fixedPhiR 	= 0
fixedPhiZ	= 0
fixedOmega 	= 1

# SOLVER PARAMETERS.
solverType	= %d
localSolver	= %d
epsilon		= %.5E
maxNewtonIter   = %d
lambda0		= %.5E
lambdaMin	= %.5E
useLowRank	= %d

# INITIAL GUESS CHECK.
max_initial_guess_checks = 0
norm_f0_target = 5.0E-05

# SWEEP CONTROL
rr_phi_max_minimum = 8.0
""" % (dr[i], dr[i], N[i], N[i], order, l, 
       N[i-1] + order, N[i-1] + order, dr[i-1], dr[i-1], order // 2, order,
       simul_directory, l, w, dr[i-1], N[i-1],
       simul_directory, l, w, dr[i-1], N[i-1],
       simul_directory, l, w, dr[i-1], N[i-1],
       simul_directory, l, w, dr[i-1], N[i-1],
       simul_directory, l, w, dr[i-1], N[i-1],
       simul_directory, l, w, dr[i-1], N[i-1],
       simul_directory, l, w, dr[i-1], N[i-1],
       solverType, localSolver, epsilon, maxNewtonIter, lambda0, lambdaMin, useLowRank)
    
    # Write and close file.
    f.write(par_file)
    f.close()

Finally, we have to write a `.pbs` script to execute on the cluster.

In [ ]:
# Some parameters for easy modification.
job_name = "ROTBOSON,l=2,Resolution_Convergence"
queue = "short"
threads = 54
# Real and virtual memory use in gigabytes.
rmem_use = 100
vmem_use = 100
# Walltime in hours.
walltime = 1
# Start executing from N[k_start].
k_start = 7

In [ ]:
torque_script = """#!/bin/bash
#
# Name of job.
#PBS -N %s
#
# Output files.
#PBS -o $PBS_JOBNAME.$PBS_JOBID.out
#PBS -e $PBS_JOBNAME.$PBS_JOBID.err
#
# Set to "%s" queue.
#PBS -q %s
#
# Resources.
#PBS -l nodes=1:ppn=%d
#PBS -l mem=%dgb
#PBS -l vmem=%dgb
#
# Walltime
#PBS -l walltime=%d:00:00
#
# Email notifications.
#PBS -m abe -M santiago.ontanon@correo.nucleares.unam.mx

# Change to current directory.
cd $PBS_O_WORKDIR

# Send email indication job start.
echo -e "Subject: $PBS_JOBNAME.$PBS_JOBID \\n\\nExecution has begun." | sendmail santiago.ontanon@correo.nucleares.unam.mx

# Job information.
echo ==================================
echo Executing on : `hostname`
echo Data: `date`
echo Directory: `pwd`
echo Assigned Resources:
echo	`cat $PBS_NODEFILE`
NPROCS=`wc -l < $PBS_NODEFILE`
echo Total: $NPROCS cpus
echo ==================================
cat $PBS_NODEFILE > $HOME/nodos
echo ==================================
echo	Output...
echo ==================================

# Set OMP_NUM_THREADS
export OMP_NUM_THREADS=$PBS_NUM_PPN
export MKL_NUM_THREADS=$PBS_NUM_PPN

# MKL OOC
export MKL_PARDISO_OOC_PATH=/storage/icn/sontanon/ROTBOSON/ooc
export MKL_PARDISO_OOC_MAX_CORE_SIZE=%d
export MKL_PARDISO_OOC_MAX_SWAP_SIZE=10000
export MKL_PARDISO_OOC_KEEP_FILE=1

""" % (job_name, queue, queue, threads, rmem_use, vmem_use, walltime, rmem_use * 1000)

# Add actual executable lines.
for i in range(k_start, len(N)):
    torque_script += """%sROTBOSON "%sl=%d,w=%.5E,dr=%.5E,N=%04d.par"\necho -e "Subject: $PBS_JOBNAME.$PBS_JOBID \\n\\nRun %04d has finished." | sendmail santiago.ontanon@correo.nucleares.unam.mx\n\n""" % (simul_directory, simul_directory, l, w, dr[i], N[i], N[i])

# Write script.
torque_script_name = "%storque_exe_script.pbs" % (local_directory)
f = open(torque_script_name, "w")
f.write(torque_script)
f.close()

## Read results

At this point, we assume that all runs have finished and we have pulled the results back to the local directory.

In [ ]:
res_dirnames = [x[0] for x in os.walk(local_directory)][1::][::-1][:]
res_dirnames

Some reordering may be necessary.

In [ ]:
res_dirnames = res_dirnames
['/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=1.00000E+00,N=0128',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=8.32797E-01,N=0154',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=6.90667E-01,N=0186',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=5.71744E-01,N=0225',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=4.73492E-01,N=0272',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=3.91831E-01,N=0329',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=3.24969E-01,N=0397',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=2.68951E-01,N=0480',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=2.22700E-01,N=0580',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=1.84342E-01,N=0701',
 '/mnt/e/RBS/Regularized Convergence/RConvergence/l=2,w=7.50000E-01,dr=1.52622E-01,N=0847']

In [ ]:
k = len(res_dirnames)
k

In [ ]:
# Quantities to examine for convergence.
res_psi_center = np.zeros(k)
res_M_1 = np.zeros(k)
res_M_2 = np.zeros(k)
res_J_1 = np.zeros(k)
res_J_2 = np.zeros(k)
res_GRV2 = np.zeros(k)
res_GRV3 = np.zeros(k)

In [ ]:
# Read from files.
for i in range(k):
    dirname = res_dirnames[i]
    res_GRV2[i], res_GRV3[i], res_psi_center[i], res_M_1[i], res_J_1[i], res_M_2[i], res_J_2[i] = (np.genfromtxt(dirname + "/GRV2.asc").item(), 
    np.genfromtxt(dirname + "/GRV3.asc").item(), 
    np.genfromtxt(dirname + "/sph_psi_f.asc", usecols=0, max_rows=1).item(),
    np.genfromtxt(dirname + "/M_Komar1.asc")[-1],
    np.genfromtxt(dirname + "/J_Komar1.asc")[-1],
    np.genfromtxt(dirname + "/M_Komar2.asc")[-1],
    np.genfromtxt(dirname + "/J_Komar2.asc")[-1])

### Convergence rates plot

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_psi_center[1:] - res_psi_center[:-1])), ".-", label = r"$\log_{10}|\Delta \psi_0|$")
#ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_GRV2[1:] - res_GRV2[:-1])), ".-", label = r"$\log_{10}|\Delta GRV_2|$")
#ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_GRV3[1:] - res_GRV3[:-1])), ".-", label = r"$\log_{10}|\Delta GRV_3|$")
ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_M_1[1:] - res_M_1[:-1])), ".-", label = r"$\log_{10}|\Delta M_1|$")
#ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_M_2[1:] - res_M_2[:-1])), ".-", label = r"$\log_{10}|\Delta M_2|$")
ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_J_1[1:] - res_J_1[:-1])), ".-", label = r"$\log_{10}|\Delta J_1|$")
#ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_J_2[1:] - res_J_2[:-1])), ".-", label = r"$\log_{10}|\Delta J_2|$")

ax.plot(-np.log10(np.abs(dr[1:k])), 4.0 * np.log10(np.abs(dr[1:k])), ".-", label = r"$4\,\log_{10}\Delta \rho$")

ax.set_xlabel(r"$-\log_{10}\,\Delta \rho$")
ax.set_title(r"Convergence rates for global quantities")

ax.legend()

plt.show()

Verify convergence with a linear fit.

In [ ]:
linregress(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_psi_center[1:] - res_psi_center[:-1])))

In [ ]:
linregress(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_M_1[1:] - res_M_1[:-1])))

In [ ]:
linregress(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(res_J_1[1:] - res_J_1[:-1])))

Now that we have affirmed that we have fourth order convergence, we can estimate the error on the last simulation.

In [ ]:
res_conv_rate = 4

In [ ]:
res_psi_center_error = (res_psi_center[-3] - res_psi_center[-2]) / ((dr[:k][-3] / dr[:k][-2])**res_conv_rate - 1.0)
res_psi_center_error

In [ ]:
res_M_1_error = (res_M_1[-3] - res_M_1[-2]) / ((dr[:k][-3] / dr[:k][-2])**res_conv_rate - 1.0)
res_M_1_error

In [ ]:
res_J_1_error = (res_J_1[-3] - res_J_1[-2]) / ((dr[:k][-3] / dr[:k][-2])**res_conv_rate - 1.0)
res_J_1_error

View value, error estimate and relative error.

In [ ]:
res_psi_center[-1], res_psi_center_error, res_psi_center_error / res_psi_center[-1]

In [ ]:
res_M_1[-1], res_M_1_error, res_M_1_error / res_M_1[-1]

In [ ]:
res_J_1[-1], res_J_1_error, res_J_1_error / res_J_1[-1]

Plot relative error estimate for each resolution.

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs((res_psi_center[:-1] - res_psi_center[1:]) / ((dr[:k][:-1] / dr[:k][1:])**res_conv_rate - 1.0))/np.abs(res_psi_center[1:])), ".-", label = r"$\log_{10}|\epsilon_R(\psi_0)|$")
ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs((res_M_1[:-1] - res_M_1[1:]) / ((dr[:k][:-1] / dr[:k][1:])**res_conv_rate - 1.0))/np.abs(res_M_1[1:])), ".-", label = r"$\log_{10}|\epsilon_R(M_1)|$")
ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs((res_J_1[:-1] - res_J_1[1:]) / ((dr[:k][:-1] / dr[:k][1:])**res_conv_rate - 1.0))/np.abs(res_J_1[1:])), ".-", label = r"$\log_{10}|\epsilon_R(J_1)|$")

ax.plot(-np.log10(np.abs(dr[1:k])), 4.0 * np.log10(np.abs(dr[1:k])), ".-", label = r"$4\,\log_{10}\Delta \rho$")

ax.set_xlabel(r"$-\log_{10}\,\Delta \rho$")
ax.set_title(r"Relative error estimates for global quantities")

ax.legend()

plt.show()

# Boundary Convergence

For boundary convergence, we have to fix a resolution size and instead of fixing $\omega$, we fix $\psi$ at a certain point.

The maximum resolution will be:

In [ ]:
bdy_dr = dr[-2]

This has been chosen so that at the least we will have $r_\infty$ at

In [ ]:
(N[0] + 2 * ghost - 1 - ghost + 0.5) * bdy_dr

With this in mind, generate rr_inf array.

In [ ]:
rr_inf = (N + 2 * ghost - 1 - ghost + 0.5) * bdy_dr
rr_inf

In [ ]:
# Directory names.
local_directory = "/mnt/e/RBS/Regularized Convergence/BConvergence/"
simul_directory = "/storage/icn/sontanon/out/BConvergence/"

# Other parameters.
order = 4
l = 2

# Solver parameters.
solverType    = 1
localSolver   = 1
epsilon       = 2.0E-13
maxNewtonIter = 50
lambda0       = 1.0E-01
lambdaMin     = 1.0E-05
useLowRank    = 0

In [ ]:
for i in range(1, len(N)):
    par_name = "%sl=%d,w=X.XXXXXE-01,dr=%.5E,N=%04d.par" % (local_directory, l, bdy_dr, N[i])
    
    f = open(par_name, "w")
    
    par_file = """
# GRID
dr 		= %.16E
dz 		= %.16E
NrInterior	= %d
NzInterior	= %d
order 		= %d

# SCALAR FIELD PROPERTIES
l	 = %d
m 	 = 1.0

# INITIAL DATA
readInitialData	= 3
NrTotalInitial 	= %d
NzTotalInitial 	= %d
dr_i		= %.16E
dz_i		= %.16E
ghost_i		= %d
order_i		= %d
log_alpha_i 	= "%sl=%d,w=X.XXXXXE-01,dr=%.5E,N=%04d/log_alpha_f.asc"
beta_i		= "%sl=%d,w=X.XXXXXE-01,dr=%.5E,N=%04d/beta_f.asc"
log_h_i		= "%sl=%d,w=X.XXXXXE-01,dr=%.5E,N=%04d/log_h_f.asc"
log_a_i		= "%sl=%d,w=X.XXXXXE-01,dr=%.5E,N=%04d/log_a_f.asc"
psi_i		= "%sl=%d,w=X.XXXXXE-01,dr=%.5E,N=%04d/psi_f.asc"
lambda_i	= "%sl=%d,w=X.XXXXXE-01,dr=%.5E,N=%04d/lambda_f.asc"
w_i		= "%sl=%d,w=X.XXXXXE-01,dr=%.5E,N=%04d/w_f.asc"

# ANALYTIC INITIAL DATA PARAMETERS.
psi0 	= 1.000
sigmaR	= 0.0
sigmaZ	= 0.0
rExt	= 0.0

# FIXED VARIABLE.
fixedPhi  	= 1
fixedPhiR 	= 2
fixedPhiZ	= 2
fixedOmega 	= 0

# SOLVER PARAMETERS.
solverType	= %d
localSolver	= %d
epsilon		= %.5E
maxNewtonIter   = %d
lambda0		= %.5E
lambdaMin	= %.5E
useLowRank	= %d

# INITIAL GUESS CHECK.
max_initial_guess_checks = 0
norm_f0_target = 5.0E-05

# SWEEP CONTROL
rr_phi_max_minimum = 8.0
""" % (bdy_dr, bdy_dr, N[i], N[i], order, l,
       N[i-1] + order, N[i-1] + order, bdy_dr, bdy_dr, order // 2, order,
       simul_directory, l, bdy_dr, N[i-1],
       simul_directory, l, bdy_dr, N[i-1],
       simul_directory, l, bdy_dr, N[i-1],
       simul_directory, l, bdy_dr, N[i-1],
       simul_directory, l, bdy_dr, N[i-1],
       simul_directory, l, bdy_dr, N[i-1],
       simul_directory, l, bdy_dr, N[i-1],
       solverType, localSolver, epsilon, maxNewtonIter, lambda0, lambdaMin, useLowRank)
    
    f.write(par_file)
    f.close()

## Read results

In [ ]:
bdy_dirnames = [x[0] for x in os.walk(local_directory)][1::][::][:]
bdy_dirnames

In [ ]:
k = len(bdy_dirnames)
k

In [ ]:
bdy_w = np.zeros(k)
bdy_M_1 = np.zeros(k)
bdy_M_2 = np.zeros(k)
bdy_J_1 = np.zeros(k)
bdy_J_2 = np.zeros(k)
bdy_GRV2 = np.zeros(k)
bdy_GRV3 = np.zeros(k)

In [ ]:
for i in range(k):
    dirname = bdy_dirnames[i]
    bdy_GRV2[i], bdy_GRV3[i], bdy_w[i], bdy_M_1[i], bdy_J_1[i], bdy_M_2[i], bdy_J_2[i] = (np.genfromtxt(dirname + "/GRV2.asc").item(), 
     np.genfromtxt(dirname + "/GRV3.asc").item(), 
     np.genfromtxt(dirname + "/w_f.asc", usecols=0, max_rows=1).item(),
     np.genfromtxt(dirname + "/M_Komar1.asc")[-1],
     np.genfromtxt(dirname + "/J_Komar1.asc")[-1],
     np.genfromtxt(dirname + "/M_Komar2.asc")[-1],
     np.genfromtxt(dirname + "/J_Komar2.asc")[-1]
    )

### Convergence rates plot

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_w[1:] - bdy_w[:-1])), ".-", label = r"$\log_{10}|\Delta \omega|$")
#ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_GRV2[1:] - bdy_GRV2[:-1])), ".-", label = r"$\log_{10}|\Delta GRV_2|$")
#ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_GRV3[1:] - bdy_GRV3[:-1])), ".-", label = r"$\log_{10}|\Delta GRV_3|$")
ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_M_1[1:] - bdy_M_1[:-1])), ".-", label = r"$\log_{10}|\Delta M_1|$")
#ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_M_2[1:] - bdy_M_2[:-1])), ".-", label = r"$\log_{10}|\Delta M_2|$")
ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_J_1[1:] - bdy_J_1[:-1])), ".-", label = r"$\log_{10}|\Delta J_1|$")
#ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_J_2[1:] - bdy_J_2[:-1])), ".-", label = r"$\log_{10}|\Delta J_2|$")

ax.set_xlabel(r"$\log_{10}\,r_\infty$")
ax.set_title(r"Convergence rates for global quantities")

ax.legend()

plt.show()

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_w[1:] - bdy_w[:-1])))

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_GRV2[1:] - bdy_GRV2[:-1])))

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_GRV3[1:] - bdy_GRV3[:-1])))

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_M_1[1:] - bdy_M_1[:-1])))

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(bdy_J_1[1:] - bdy_J_1[:-1])))

In [ ]:
bdy_w_error = (bdy_w[-2] - bdy_w[-1]) / ((rr_inf[:k][-1] / rr_inf[:k][-2])**2 - 1.0)
bdy_w_error

In [ ]:
bdy_M_1_error = (bdy_M_1[-2] - bdy_M_1[-1]) / ((rr_inf[:k][-1] / rr_inf[:k][-2])**1 - 1.0)
bdy_M_1_error

In [ ]:
bdy_J_1_error = (bdy_J_1[-2] - bdy_J_1[-1]) / ((rr_inf[:k][-1] / rr_inf[:k][-2])**2 - 1.0)
bdy_J_1_error

In [ ]:
bdy_w[-1], bdy_w_error, bdy_w_error / bdy_w[-1]

In [ ]:
bdy_M_1[-1], bdy_M_1_error, bdy_M_1_error / bdy_M_1[-1]

In [ ]:
bdy_J_1[-1], bdy_J_1_error, bdy_J_1_error / bdy_J_1[-1]

## Error comparison

In [ ]:
res_M_1_error / res_M_1[-1], bdy_M_1_error / bdy_M_1[-1]

In [ ]:
res_J_1_error / res_J_1[-1], bdy_J_1_error / bdy_J_1[-1]